# YOLO 얼굴 검출 — 학습 & 하이퍼파라미터 비교

1. 데이터셋 확인 · 2. YOLO 학습 · 3. `experiments/yolo_runs.jsonl` 기록 · 4. 비교

설치: `pip install -r requirements-notebook.txt`

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "config.py").is_file():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("emotion-bot 루트(config.py)를 찾지 못했습니다.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os

import config

os.chdir(ROOT)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import yolo_dataset_utils

EXPERIMENTS_LOG = ROOT / "experiments" / "yolo_runs.jsonl"
EXPERIMENTS_LOG.parent.mkdir(parents=True, exist_ok=True)
NOTEBOOK_SETUP_VERSION = "yolo_nb_setup_v4"
print("ROOT:", ROOT)
print("cwd:", Path.cwd())
print("setup:", NOTEBOOK_SETUP_VERSION)
print("data.yaml:", config.YOLO_DATA_YAML.resolve())
print("yolo_dataset_utils:", getattr(yolo_dataset_utils, "__file__", "MISSING"))

## 1. 데이터셋 확인

Human Face (Roboflow) 연동:

```bash
python scripts/link_human_face_dataset.py
```

원본: `C:\Users\moonjintae\datasets\Human Face dataset` → `data/face_detect/data.yaml`이 가리킴

**주의:** 출력이 `data.yaml: True` / `train: 0 val: 0` 이면 **디스크의 ipynb가 아닌 예전 셀**을 실행 중입니다.  
→ 노트북 탭에서 **파일 다시 로드(Reload/Revert)** 후 셀 1·아래 셀을 다시 실행하세요. 정상이면 `=== dataset check v4 ===` 와 `train: 800 images` 가 보입니다.

In [ ]:
# 셀 1 없이 단독 실행해도 동작 (구버전 data.yaml: True / train: 0 val: 0 방지)
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import yaml

_nb_root = Path.cwd().resolve()
for _ in range(5):
    if (_nb_root / "config.py").is_file():
        break
    if _nb_root.parent == _nb_root:
        break
    _nb_root = _nb_root.parent
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))
_scripts = _nb_root / "scripts"
if str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))

import config
import yolo_dataset_utils

importlib.reload(yolo_dataset_utils)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
print("=== dataset check v4 ===")
print("notebook root:", _nb_root)

yaml_path = config.YOLO_DATA_YAML.resolve()
print("data.yaml path:", yaml_path)
print("data.yaml exists:", yaml_path.is_file())

if not yaml_path.is_file():
    print("→ python scripts/link_human_face_dataset.py")
else:
    yolo_dataset_utils.print_dataset_report(
        yaml_path,
        fallback=config.HUMAN_FACE_DATASET,
    )

## 2. 파라미터

In [ ]:
YAML_CFG = ROOT / "configs" / "yolo" / "example.yaml"
cfg = yaml.safe_load(YAML_CFG.read_text(encoding="utf-8")) if YAML_CFG.is_file() else {}
TRAIN_PARAMS = {
    "run_name": cfg.get("run_name", "face_train_nb"),
    "base": cfg.get("base", config.YOLO_BASE_ARCH),
    "epochs": cfg.get("epochs", 30),
    "batch": cfg.get("batch", config.YOLO_TRAIN_BATCH),
    "imgsz": cfg.get("imgsz", config.YOLO_IMGSZ),
    "patience": cfg.get("patience", config.YOLO_TRAIN_PATIENCE),
    "device": cfg.get("device", None),
    "resume": False,
}
TRAIN_PARAMS

## 3. 학습 & export → weights/face_yolo.pt

In [ ]:
def export_best(run_dir: Path, dest: Path) -> Path:
    shutil.copy2(run_dir / "best.pt", dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    return dest

def extract_metrics(results) -> dict:
    metrics = {}
    rd = getattr(results, "results_dict", None) or {}
    for key in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        if key in rd:
            metrics[key.split("/")[-1]] = float(rd[key])
    return metrics

def log_run(params, metrics, save_dir, notes=""):
    record = {
        "run_id": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
        "params": params,
        "metrics": metrics,
        "save_dir": str(save_dir),
        "notes": notes,
    }
    with EXPERIMENTS_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record

if not config.YOLO_DATA_YAML.is_file():
    raise FileNotFoundError(config.YOLO_DATA_YAML)

model = YOLO(TRAIN_PARAMS["base"])
kwargs = dict(
    data=str(config.YOLO_DATA_YAML.resolve()),
    epochs=TRAIN_PARAMS["epochs"],
    batch=TRAIN_PARAMS["batch"],
    imgsz=TRAIN_PARAMS["imgsz"],
    patience=TRAIN_PARAMS["patience"],
    project=str(config.YOLO_TRAIN_PROJECT),
    name=TRAIN_PARAMS["run_name"],
    device=TRAIN_PARAMS["device"],
    exist_ok=True,
)
if TRAIN_PARAMS.get("resume"):
    kwargs["resume"] = True
results = model.train(**kwargs)
save_dir = Path(results.save_dir)
metrics = extract_metrics(results)
exported = export_best(save_dir / "weights", config.YOLO_MODEL)
record = log_run(TRAIN_PARAMS, metrics, save_dir)
print("exported:", exported, "metrics:", metrics)
record

## 4. 학습 곡선

In [ ]:
png = save_dir / "results.png"
if png.is_file():
    display(Image(filename=str(png)))
else:
    print("no results.png")

## 5. run 비교

In [ ]:
def load_log():
    if not EXPERIMENTS_LOG.is_file() or EXPERIMENTS_LOG.stat().st_size == 0:
        return pd.DataFrame()
    rows = [json.loads(ln) for ln in EXPERIMENTS_LOG.read_text(encoding="utf-8").splitlines() if ln.strip()]
    flat = []
    for r in rows:
        row = {"run_id": r["run_id"]}
        row.update({f"p_{k}": v for k, v in r.get("params", {}).items()})
        row.update({f"m_{k}": v for k, v in r.get("metrics", {}).items()})
        flat.append(row)
    return pd.DataFrame(flat)

df = load_log()
display(df.sort_values("run_id") if not df.empty else "yolo_runs.jsonl 비어 있음")
if not df.empty:
    mcols = [c for c in df.columns if c.startswith("m_")]
    if mcols:
        df.plot(x="run_id", y=mcols, kind="bar", figsize=(8, 4), rot=45)
        plt.tight_layout()
        plt.show()

## 6. (선택) 하이퍼파라미터 스윕

In [ ]:
RUN_SWEEP = False
SWEEP_GRID = [
    {"run_name": "sweep_b8", "batch": 8, "epochs": 20},
    {"run_name": "sweep_b16", "batch": 16, "epochs": 20},
]
if RUN_SWEEP:
    for ov in SWEEP_GRID:
        p = {**TRAIN_PARAMS, **ov}
        m = YOLO(p["base"])
        res = m.train(data=str(config.YOLO_DATA_YAML.resolve()), project=str(config.YOLO_TRAIN_PROJECT), exist_ok=True, **{k: p[k] for k in ("epochs", "batch", "imgsz", "patience", "device")}, name=p["run_name"])
        log_run(p, extract_metrics(res), Path(res.save_dir), notes="sweep")
    display(load_log().tail(len(SWEEP_GRID)))

## 7. (선택) 샘플 검출

In [ ]:
cfg = yaml.safe_load(config.YOLO_DATA_YAML.read_text(encoding="utf-8"))
ds_root = Path(cfg["path"])
val_dir = ds_root / cfg["val"]
sample = next(val_dir.glob("*"), None) if val_dir.is_dir() else None
if sample and config.YOLO_MODEL.is_file():
    pred = YOLO(str(config.YOLO_MODEL)).predict(source=str(sample), conf=config.YOLO_CONF)
    out = ROOT / "experiments" / "preview"
    out.mkdir(parents=True, exist_ok=True)
    pred[0].save(filename=str(out / "yolo_preview.jpg"))
    display(Image(filename=str(out / "yolo_preview.jpg")))
else:
    print("샘플 또는 face_yolo.pt 없음")